# 5. Learn a Neo-Hookean material from data

Train a neural energy density for one compressible Neo-Hookean material with
$(\lambda,\mu)=(2,3)$. Native Neo-Hookean generates the energy and stress
labels. PyTorch differentiates the learned energy to get stress and tangent
stiffness.

Check the network on unseen deformations, then export its weights for the
equilibrium fit in lesson 6. In Try it, extend a loading path to test
extrapolation in compression and tension.

In [ ]:
from copy import deepcopy
from pathlib import Path
from tempfile import TemporaryDirectory

import matplotlib.pyplot as plt
import numpy as np
import torch
from torch import nn
%matplotlib inline
from litepgo import fem

torch.set_num_threads(1)
torch.manual_seed(42)
dtype = torch.float64
reference_theta = np.array([2., 3.])

## Formulate the neural energy density

### Encode the elastic deformation

As in lesson 1, set $F_e=FF_p^{-1}$. Define $C=F_e^\mathsf{T}F_e$ and
$J=\det F_e>0$. The input to the network is the three-component vector

$$z=\big(J^{-2/3}I_1-3,\ J^{-4/3}I_2-3,\ \log J\big),\qquad
I_1=\operatorname{tr}C,\quad
I_2=\tfrac12\big(I_1^2-\operatorname{tr}(C^2)\big).$$

The first two entries describe shape changes after removing uniform volume
scaling. The third measures volume change. For a uniform stretch $F_e=sI$,
the first two entries are zero and the last is $3\log s$. Rigid rotations
leave the three inputs unchanged. The features therefore encode objectivity
and isotropy before training starts.

At the relaxed state $F_e=I$, we have $C=I$, $J=1$, and $I_1=I_2=3$.
For any small perturbation $\delta F_e=H$, the first variations are

$$\delta C=H+H^\mathsf{T},\qquad
\delta J=\operatorname{tr}H,\qquad
\delta I_1=2\operatorname{tr}H.$$

Differentiating $I_2$ gives
$\delta I_2=I_1\delta I_1-\operatorname{tr}(C\,\delta C)
=4\operatorname{tr}H$ at $C=I$. The product rule then gives

$$\begin{aligned}
\delta z_1 &= \delta I_1-2\delta J=0,\\
\delta z_2 &= \delta I_2-4\delta J=0,\\
\delta z_3 &= \delta\log J=\operatorname{tr}H.
\end{aligned}$$

The first two features can still change at higher order. Check their
first derivatives at $I$ with autograd. The derivative of $\log J$ at
$I$ is the identity matrix because its directional derivative is
$\operatorname{tr}H$.

In [ ]:
def invariants(Fe):
    J = torch.linalg.det(Fe)
    C = Fe.transpose(-1, -2) @ Fe
    I1 = C.diagonal(dim1=-2, dim2=-1).sum(-1)
    I2 = 0.5 * (I1.square() - C.square().sum(dim=(-2, -1)))
    return torch.stack(
        (J.pow(-2 / 3) * I1 - 3, J.pow(-4 / 3) * I2 - 3, J.log()), dim=-1,
    )

rest_Fe = torch.eye(3, dtype=dtype)
feature_gradients = torch.autograd.functional.jacobian(invariants, rest_Fe)
zero = torch.zeros_like(rest_Fe)
expected_gradients = torch.stack((zero, zero, rest_Fe))
torch.testing.assert_close(feature_gradients, expected_gradients, rtol=0, atol=1e-12)
print('Feature gradient norms at rest:', feature_gradients.flatten(1).norm(dim=1))

### Map the three features to a stress-free energy

Two hidden layers have 32 units each. The output is one scalar
$f_\omega(z)$. All 1,216 network parameters are trained. Softplus
($\log(1+\exp x)$ in its default form) gives a smooth activation so we
can differentiate energy for stress and differentiate stress for stiffness.

The final density is

$$W(F,F_p)=\det(F_p)\left[
f_\omega(z)-f_\omega(0)
-\partial_{z_3}f_\omega(0)\,\log J
\right].$$

At $F_e=I$, all three features are zero. Subtracting $f_\omega(0)$ sets
the energy there to zero. For the stress, hold $F_p$ fixed and define
$a=\partial_{z_3}f_\omega(0)$. The chain rule and the feature derivatives
above give

$$\delta f_\omega(z)\big|_{F_e=I}
=\sum_{i=1}^3\partial_{z_i}f_\omega(0)\,\delta z_i
=a\,\delta\log J.$$

The variation of the correction $-a\log J$ cancels exactly that term.
Since $\det(F_p)$ is constant in this derivative, $\delta W=0$ for every
perturbation of $F$. Thus $P=\partial W/\partial F=0$ at the relaxed
state. The construction leaves higher-order responses available to learn,
including the stiffness at rest.

In the code, `origin_gradient[2]` is $a$. The forward pass needs this
slope even for a caller that only wants the value of $W$. The two context
managers temporarily leave inference mode and enable gradient recording
so `autograd.grad` can evaluate the slope at `origin`.

`build_graph` captures the caller's gradient setting before that temporary
change. With gradient recording enabled by the caller, `create_graph=True`
keeps the slope differentiable with respect to the network weights. In a
value-only call under `no_grad` or `inference_mode`, the block computes the
slope as a number with `create_graph=False`. The caller's gradient setting
resumes after the block.

The factor $\det(F_p)$ expresses $W$ per original reference volume.

In [ ]:
class NeuralEnergy(nn.Module):
    def __init__(self, features=invariants):
        super().__init__()
        self.features = features
        self.net = nn.Sequential(
            nn.Linear(3, 32),
            nn.Softplus(),
            nn.Linear(32, 32),
            nn.Softplus(),
            nn.Linear(32, 1, bias=False),
        ).double()

    def forward(self, F, Fp):
        Fe = F @ torch.linalg.inv(Fp)
        z = self.features(Fe)
        build_graph = torch.is_grad_enabled()
        # Value-only calls also need the derivative at the reference point.
        with torch.inference_mode(False), torch.enable_grad():
            origin = torch.zeros(3, dtype=F.dtype, device=F.device, requires_grad=True)
            f0 = self.net(origin).squeeze(-1)
            origin_gradient, = torch.autograd.grad(f0, origin, create_graph=build_graph)

        elastic_energy = self.net(z).squeeze(-1) - f0 - origin_gradient[2] * z[..., 2]
        return torch.linalg.det(Fp) * elastic_energy

### Architecture diagram

The blue layers carry trainable parameters. Feature extraction and the
reference correction are deterministic operations in `NeuralEnergy.forward`.

In [ ]:
model = NeuralEnergy()
print(model.net)
print('Trainable parameters:', sum(parameter.numel() for parameter in model.parameters()))

stages = (
    ('$F, F_p$\nElastic $F_e$', False),
    ('3 invariants\n$z_1, z_2, \\log J$', False),
    ('Linear 3 → 32\nSoftplus', True),
    ('Linear 32 → 32\nSoftplus', True),
    ('Linear 32 → 1\n$f_\\omega(z)$', True),
    ('Reference correction\n$W(F,F_p)$', False),
)
fig, ax = plt.subplots(figsize=(12, 2.2), layout='constrained')
centers = np.linspace(.085, .915, len(stages))
for left, right in zip(centers[:-1], centers[1:]):
    ax.annotate('', xy=(right - .071, .58), xytext=(left + .071, .58),
                xycoords='axes fraction', arrowprops={'arrowstyle': '->', 'color': '#475569'})
for x, (label, trained) in zip(centers, stages):
    ax.text(x, .58, label, transform=ax.transAxes, ha='center', va='center',
            bbox={'boxstyle': 'round,pad=.5', 'facecolor': '#dbeafe' if trained else '#f1f5f9',
                  'edgecolor': '#64748b'})
ax.text(.58, .19, 'Blue layers are trained · 1,216 parameters', transform=ax.transAxes,
        ha='center', color='#334155')
ax.set_axis_off()
plt.show()

## Generate native material data

### Sample proper rotations with QR

Draw a batch of $3\times3$ matrices with independent standard normal entries.
For each matrix, QR factorization gives an orthogonal $Q$ with
$Q^\mathsf{T}Q=I$. Its determinant can be $+1$ or $-1$. Multiplying the last
column by $\det Q$ keeps the columns orthonormal and makes the new
determinant $(\det Q)^2=1$. Each output is therefore a proper rotation.

This sampler uses the orientations produced by NumPy's QR sign convention.
Uniform sampling on $SO(3)$ would also require correcting the signs from
the diagonal of $R$. Here we check orthogonality and the positive determinant.

In [ ]:
def sample_rotations(rng, count):
    matrices = rng.normal(size=(count, 3, 3))
    Q, _ = np.linalg.qr(matrices)
    Q[:, :, 2] *= np.linalg.det(Q)[:, None]
    return Q

check_Q = sample_rotations(rng=np.random.default_rng(0), count=4)
np.testing.assert_allclose(
    check_Q.transpose(0, 2, 1) @ check_Q, np.broadcast_to(np.eye(3), check_Q.shape),
    atol=1e-12,
)
np.testing.assert_allclose(np.linalg.det(check_Q), 1., atol=1e-12)

### Choose how much to stretch and in which directions

Each row of `stretches` contains three independently sampled principal
stretch ratios from $[0.7,1.3]$. For example, $(0.8,1.1,1.2)$ means a
20% compression along one principal direction and 10% and 20% stretches
along the other two. Draw two independent rotations per row and form

$$F=Q_1\operatorname{diag}(s_1,s_2,s_3)Q_2^\mathsf{T}.$$

$Q_2^\mathsf{T}$ expresses a vector in the principal directions, the diagonal
matrix stretches those components, and $Q_1$ rotates the result into its
output directions. The expression `Q1 * stretches[:, None, :]` scales
each column of `Q1` by its corresponding stretch, equivalent to multiplying
on the right by the diagonal matrix. Rotations preserve singular values,
so the singular values of $F$ are the three sampled stretches. All samples
have $\det F=s_1s_2s_3>0$.

In [ ]:
def sample_deformations(count, seed):
    rng = np.random.default_rng(seed)
    stretches = rng.uniform(.7, 1.3, size=(count, 3))
    Q1 = sample_rotations(rng=rng, count=count)
    Q2 = sample_rotations(rng=rng, count=count)
    scaled_Q1 = Q1 * stretches[:, None, :]
    return scaled_Q1 @ Q2.transpose(0, 2, 1)

example_F = sample_deformations(count=1, seed=0)
example_stretches = np.linalg.svd(example_F, compute_uv=False)[0]
assert np.all((example_stretches >= .7) & (example_stretches <= 1.3))
assert np.linalg.det(example_F)[0] > 0
print('Example principal stretches:', example_stretches)

### Label each deformation and keep the splits separate

Use $F_p=I$ and the same $(\lambda,\mu)=(2,3)$ for every native call.
For each $F_i$, native Neo-Hookean returns a scalar energy $W_i$ and a
$3\times3$ first Piola stress $P_i$. The training set uses 10,000 samples
from seed 11 to update weights. The validation set uses 2,000 samples from
seed 12 to select weights. The later generalization section draws an
independent random test set.

For evaluation we can also request the native tangent. Its flattened matrix
uses column-major indices on both axes. For example, $F_{01}$ has flattened
index 3 in that order and index 1 in Torch's row-major order. Reorder both
axes before comparing native and Torch tangents.

In [ ]:
def native_labels(F, *, tangent=False):
    Fp = np.eye(3)
    W = torch.tensor([fem.NeoHookean.eval_W(f, Fp, reference_theta) for f in F], dtype=dtype)
    P = torch.from_numpy(np.array([fem.NeoHookean.eval_dW_dF(f, Fp, reference_theta) for f in F]))
    if not tangent:
        return W, P

    row_major_order = [0, 3, 6, 1, 4, 7, 2, 5, 8]
    indices = np.ix_(row_major_order, row_major_order)
    H = [fem.NeoHookean.eval_d2W_dF2(f, Fp, reference_theta)[indices] for f in F]
    return W, P, torch.from_numpy(np.array(H))

def labeled_samples(count, seed):
    F = sample_deformations(count=count, seed=seed)
    W, P = native_labels(F)
    return torch.from_numpy(F), W, P

train_data = labeled_samples(count=10_000, seed=11)
validation_data = labeled_samples(count=2_000, seed=12)

## Differentiate the learned energy

The network predicts one energy $\widehat W$ per sample. As in lesson 1,
differentiate their sum with respect to $F$ to get each sample's stress
$\widehat P=\partial\widehat W/\partial F$.

Training compares $\widehat P$ with a native stress label. That error needs
a derivative with respect to the network weights $\omega$. The relevant
quantity is the mixed derivative

$$\frac{\partial \widehat P}{\partial\omega}
=\frac{\partial}{\partial\omega}
  \left(\frac{\partial \widehat W}{\partial F}\right).$$

For example, if $\widehat W=\omega x^2$, then $\widehat P=2\omega x$.
The stress is already a derivative with respect to $x$, and it still
depends on $\omega$. In `prediction(..., training=True)`, `create_graph=True`
records the operations that produce the stress so the loss can follow
that dependency back to the weights.

Tangent evaluation also retains this derivative graph. It differentiates
each stress component with respect to $F$ to get $\partial P/\partial F$.

In [ ]:
def prediction(model, F, *, training=False, tangent=False):
    F = F.detach().requires_grad_()
    Fp = torch.eye(3, dtype=F.dtype, device=F.device).expand_as(F)
    W = model(F, Fp)
    P, = torch.autograd.grad(W.sum(), F, create_graph=training or tangent)
    if not tangent:
        return W, P

    rows = []
    flat_stress = P.flatten(1)
    for i in range(9):
        row, = torch.autograd.grad(flat_stress[:, i].sum(), F, retain_graph=i < 8)
        rows.append(row.flatten(1))
    return W, P, torch.stack(rows, dim=1)

## Train on native energy and stress labels

### Define the normalized loss

Compute $s_W$ and $s_P$ as the RMS values of the native energy labels and
native stress components in the training set. Keep them fixed for training
and validation. The objective is the sum of two mean squared errors,

$$L_W=\frac1N\sum_i\left(\frac{\widehat W_i-W_i}{s_W}\right)^2,\qquad
L_P=\frac1{9N}\sum_i\frac{\|\widehat P_i-P_i\|_F^2}{s_P^2},\qquad
L=L_W+L_P.$$

The energy tensor has one entry per sample. The stress tensor has nine,
so its `mean()` divides by $9N$. Below, take the stress term on its own
and differentiate it with respect to the network weights. This checks
that the stress error provides a training signal on its own.

In [ ]:
scales = tuple(labels.square().mean().sqrt() for labels in train_data[1:])

def loss_terms(model, data, scales, *, training=False):
    F, W, P = data
    energy_scale, stress_scale = scales
    predicted_W, predicted_P = prediction(model, F, training=training)
    energy_error = ((predicted_W - W) / energy_scale).square().mean()
    stress_error = ((predicted_P - P) / stress_scale).square().mean()
    return energy_error, stress_error

probe_batch = tuple(values[:4] for values in train_data)
_, stress_error = loss_terms(model, probe_batch, scales, training=True)
weight_gradients = torch.autograd.grad(stress_error, tuple(model.parameters()))
stress_gradient_norm = torch.cat([gradient.flatten() for gradient in weight_gradients]).norm()
assert torch.isfinite(stress_gradient_norm) and stress_gradient_norm > 0
print(f'Stress-only loss gradient norm: {stress_gradient_norm.item():.6g}')

### Update and select the network weights

Both optimizers minimize the same objective, $L_W+L_P$. We start with
inexpensive minibatch updates and then use full-batch updates to refine
the same weights. Before training, record the initial validation loss.
Whenever we check validation, save a copy of the weights if the loss
improves. The validation samples are used only to score those weights.

In [ ]:
model.train().requires_grad_(True)
checkpoint = {'loss': float('inf'), 'weights': None}

def validate(model, checkpoint, stage, step):
    value = sum(loss_terms(model, validation_data, scales)).item()
    if value < checkpoint['loss']:
        checkpoint.update(loss=value, weights=deepcopy(model.state_dict()))
    print(f'{stage} {step}: validation loss = {value:.6g}')
    return value

initial_validation = validate(model, checkpoint, 'Initial', 0)

#### Start with minibatch Adam

Adam keeps two moving averages for each weight. The average gradient
$m_t$ smooths the direction across minibatches. The average squared
gradient $v_t$ sets the scale for that weight. With the bias-corrected
averages $\widehat m_t$ and $\widehat v_t$, the update is elementwise,

$$\omega_{t+1}=\omega_t-\eta\,\frac{\widehat m_t}{\sqrt{\widehat v_t}+\epsilon}.$$

From the initialized weights, take 1,500 steps with 512 randomly drawn
samples per step. Each step clears the old gradients, computes the
minibatch gradient with `backward()`, and lets `adam.step()` update the
weights using its running averages. Processing a small batch makes each
of these updates inexpensive.

In [ ]:
def run_adam(model, checkpoint):
    train_F, train_W, train_P = train_data
    generator = torch.Generator().manual_seed(42)
    adam = torch.optim.Adam(model.parameters(), lr=3e-3)
    for step in range(1, 1501):
        batch = torch.randint(len(train_F), (512,), generator=generator)
        adam.zero_grad()
        batch_data = (train_F[batch], train_W[batch], train_P[batch])
        value = sum(loss_terms(model, batch_data, scales, training=True))
        value.backward()
        adam.step()
        if step % 100 == 0:
            validate(model, checkpoint, 'Adam', step)

run_adam(model, checkpoint)

#### Refine with full-batch L-BFGS

L-BFGS uses recent pairs of weight changes $\Delta\omega$ and gradient
changes $\Delta g$ to approximate the curvature of the loss in weight
space. Locally, $\Delta g\approx (\nabla_\omega^2 L)\Delta\omega$.
This gives information about how changing one weight can affect the
gradient of another. The name *limited memory* refers to keeping only
a finite history of those pairs. The optimizer uses the approximation
to choose a direction, then a line search chooses the step length.

Start from the weights at the end of Adam. Now every gradient uses all
10,000 training samples, so successive gradients measure the effect of
moving the weights on the same objective. `lbfgs.step(closure)` can call
`closure` several times to try step lengths. Each call clears gradients
and recomputes the full loss and its gradient at the current weights.
These evaluations are more expensive, so we use them for refinement
after Adam has already reduced the loss. Each call to `step` allows up
to 20 iterations, for a total budget of 300.

In [ ]:
def run_lbfgs(model, checkpoint):
    lbfgs = torch.optim.LBFGS(
        model.parameters(), lr=1., max_iter=20, history_size=100,
        tolerance_grad=1e-10, tolerance_change=1e-14, line_search_fn='strong_wolfe',
    )

    def closure():
        lbfgs.zero_grad()
        value = sum(loss_terms(model, train_data, scales, training=True))
        value.backward()
        return value

    for budget in range(20, 301, 20):
        lbfgs.step(closure)
        validate(model, checkpoint, 'L-BFGS budget', budget)

run_lbfgs(model, checkpoint)

#### Restore the best validation weights

Restore the weights from the lowest validation loss observed across
both stages. Freeze the weights for the generalization tests, where we
will still differentiate the learned energy with respect to deformation.

In [ ]:
best_validation = checkpoint['loss']
assert best_validation < initial_validation
model.load_state_dict(checkpoint['weights'])
model.eval().requires_grad_(False)
print('Best validation loss:', best_validation)

## Check generalization to unseen deformations

### Choose the test deformations

Use the weights selected by validation and keep them fixed. The 10,000 training
samples update the weights, the 2,000 validation samples select the weights,
and the 512 random test samples measure accuracy on a fresh draw. The random
sets use the same sampling rule with different seeds. Each deformation has
three principal stretches sampled from $[0.7,1.3]$ and random orientations.
Every comparison uses $F_p=I$ and the same native Neo-Hookean material.

The four structured paths also stay within the training range of principal
stretches. They sweep one loading mode at a time, with 61 points per path.
For example, the uniaxial path uses $F=\operatorname{diag}(s,1,1)$ with
$s\in[0.7,1.3]$. In Try it, we extend this path beyond both ends of the
training interval.

In [ ]:
test_F = sample_deformations(count=512, seed=13)

def diagonal_path(x, y, z):
    F = np.zeros((len(x), 3, 3))
    F[:, range(3), range(3)] = np.column_stack((x, y, z))
    return F

def uniaxial_path(stretches):
    ones = np.ones_like(stretches)
    return diagonal_path(stretches, ones, ones)

test_stretch = np.linspace(.7, 1.3, 61)
ones = np.ones_like(test_stretch)
shear = np.repeat(np.eye(3)[None], len(test_stretch), axis=0)
shear[:, 0, 1] = np.linspace(-.5, .5, len(shear))
inside_cases = {
    'Random test': test_F,
    'Uniaxial': uniaxial_path(test_stretch),
    'Biaxial': diagonal_path(test_stretch, test_stretch, ones),
    'Volumetric': diagonal_path(test_stretch, test_stretch, test_stretch),
    'Shear': shear,
}

### Measure the relative error over a batch

For the same batch of $F$, `native_labels` gives reference values and
`prediction` gives learned values. With `tangent=True`, both return $W$,
$P$, and $H=\partial P/\partial F=\partial^2 W/\partial F^2$. The tangent
measures how stress changes when $F$ changes. Training used $W$ and $P$
labels, so this also checks how accurately the network learned the
curvature of the energy.

For each output, take the norm of the difference over all samples and
components, then divide by the norm of the native values over those same
entries. For example, the stress error is

$$\operatorname{err}_P =
\frac{\sqrt{\sum_{n=1}^{N}\sum_{i,j}(\widehat P_{n,ij}-P^{\mathrm{native}}_{n,ij})^2}}
     {\sqrt{\sum_{n=1}^{N}\sum_{i,j}(P^{\mathrm{native}}_{n,ij})^2}}.$$

In the code, `.norm()` treats the whole tensor as one vector. Each sample
contributes one entry for $W$, nine for $P$, and 81 for the full $9\times9$
material tangent $H$. This gives one percentage per quantity and per test
case. Even when an individual sample has zero reference stress, the
normalization uses the norm of the entire reference batch.

In [ ]:
def relative_errors(model, F):
    native = native_labels(F, tangent=True)
    learned = prediction(model, torch.from_numpy(F), tangent=True)
    return tuple(
        ((actual.detach() - expected).norm() / expected.norm()).item()
        for actual, expected in zip(learned, native)
    )

### Compare the test errors

Report the three errors separately for each case. For the range covered by
training, check that the energy and stress errors are below 1% and the
material tangent error is below 2%. The in-range uniaxial row gives a
baseline for the extrapolation exercise.

In [ ]:
print(f'{"Test case":<29} {"W error":>10} {"P error":>10} {"Tangent":>10}')
for label, F in inside_cases.items():
    energy_error, stress_error, tangent_error = relative_errors(model, F)
    assert energy_error < .01 and stress_error < .01 and tangent_error < .02
    print(f'{label:<29} {energy_error:>10.3%} {stress_error:>10.3%} {tangent_error:>10.3%}')

### Verify rest and rotation behavior

At $F=I$, check that both $W$ and $P$ vanish. Then apply a rigid rotation
$Q$ to four random test samples. The energy should stay the same and the
stress should rotate with the deformation, so $P(QF)=Q P(F)$.

In [ ]:
rest = torch.eye(3, dtype=dtype)[None]
W_rest, P_rest = prediction(model, rest)
torch.testing.assert_close(W_rest, torch.zeros_like(W_rest), atol=1e-12, rtol=0)
torch.testing.assert_close(P_rest, torch.zeros_like(P_rest), atol=1e-12, rtol=0)

Q = torch.tensor([[0., -1., 0.], [1., 0., 0.], [0., 0., 1.]], dtype=dtype)
sample = torch.from_numpy(test_F[:4])
W, P = prediction(model, sample)
W_rotated, P_rotated = prediction(model, Q @ sample)
torch.testing.assert_close(W_rotated, W, atol=1e-12, rtol=1e-10)
torch.testing.assert_close(P_rotated, Q @ P, atol=1e-12, rtol=1e-10)
print('Reference state and rotation checks passed.')

## Export the weights for lesson 6

Flatten the trained network's weights and biases in layer order
$(w_1,b_1,w_2,b_2,w_3)$. The two hidden layers contribute
$32\cdot3+32$ and $32\cdot32+32$ entries. The output layer contributes
32, giving 1,216 entries in all. Lesson 6 reads this vector from
`trained_theta.npy`.

In [ ]:
trained_theta = torch.nn.utils.parameters_to_vector(model.parameters()).detach().numpy()
assert trained_theta.shape == (1216,)

lesson_dir = Path('tutorials/torch') if Path('tutorials/torch').is_dir() else Path('.')
checkpoint_path = lesson_dir / 'trained_theta.npy'
with TemporaryDirectory(dir=lesson_dir) as directory:
    temporary = Path(directory) / checkpoint_path.name
    np.save(temporary, trained_theta, allow_pickle=False)
    temporary.replace(checkpoint_path)
print(f'Saved {trained_theta.size} trained parameters to {checkpoint_path.name}')

## Try it

### Test the original network at extreme stretches

Keep the trained weights fixed and follow the uniaxial path
$F=\operatorname{diag}(s,1,1)$ from $s=0.25$ to $s=2.0$. The first
endpoint is 75% compression along the first axis, and the second is
100% extension. The other two stretches stay at 1.

Compare the relative errors in $W$, $P$, and the material tangent
in strong compression, inside the training range, and in strong
tension. Where do the learned curves start to separate from native
Neo-Hookean? Does the tangent remain accurate as far as the energy does?

### Measure errors at both ends of the path

The principal stretch $s$ goes below $0.7$ in compression and above
$1.3$ in tension, so both ends require extrapolation. The determinant
stays positive because $\det F=s$. Evaluate separate batches in strong
compression ($0.25$ to $0.50$), the training range ($0.70$ to $1.30$),
and strong tension ($1.75$ to $2.00$). Each row uses all the components
of $W$, $P$, and the material tangent for that batch.

In [ ]:
stretch_ranges = {
    'Strong compression': np.linspace(.25, .50, 61),
    'Training range': test_stretch,
    'Strong tension': np.linspace(1.75, 2.0, 61),
}

print(f'{"Uniaxial test":<22} {"W error":>10} {"P error":>10} {"Tangent":>10}')
baseline_errors = {}
for label, stretches in stretch_ranges.items():
    errors = relative_errors(model, uniaxial_path(stretches))
    baseline_errors[label] = errors
    W_error, P_error, H_error = errors
    print(f'{label:<22} {W_error:>10.3%} {P_error:>10.3%} {H_error:>10.3%}')

### Compare the learned and native curves

Plot $W$, the axial stress $P_{11}$, and the axial tangent
$\partial P_{11}/\partial F_{11}$ along the full path. These are the
energy curve and its first two derivatives along $s$. The shaded regions
mark both extrapolation ranges, so the plots show where the curves
separate as we move away from the training interval.

In [ ]:
stretch = np.linspace(.25, 2.0, 351)
F_path = uniaxial_path(stretch)
W_native, P_native, H_native = native_labels(F_path, tangent=True)
W_learned, P_learned, H_learned = prediction(
    model, torch.from_numpy(F_path), tangent=True,
)
curves = (
    (r'Energy $W$', W_native, W_learned),
    (r'Axial stress $P_{11}$', P_native[:, 0, 0], P_learned[:, 0, 0]),
    (r'Axial tangent $\partial P_{11}/\partial F_{11}$', H_native[:, 0, 0], H_learned[:, 0, 0]),
)
fig, axes = plt.subplots(1, 3, figsize=(11, 3.6), layout='constrained')
for ax, (label, native, learned) in zip(axes, curves):
    ax.plot(stretch, native.numpy(), label='Native')
    ax.plot(stretch, learned.detach().numpy(), '--', label='Learned')
    ax.axvspan(stretch[0], .7, color='#fbbf24', alpha=.18, label='Extrapolation')
    ax.axvspan(1.3, stretch[-1], color='#fbbf24', alpha=.18)
    ax.set(xlabel='Axial stretch', ylabel=label, xlim=(stretch[0], stretch[-1]))
    ax.grid(alpha=.25)
axes[0].legend()
plt.show()

In this run, both extrapolation batches have larger errors than the in-range
batch. Strong compression is the harder test. Near $s=0.25$, the learned
energy curve still looks close on the plot, but the learned compressive
stress magnitude and axial tangent are both too large. The relative error
of the full material tangent exceeds 100% over the strong-compression batch.

### Give the network a prior for large deformations

Can the input features help the network extrapolate? Build a second
model that sees a shape feature on the scale of squared stretches and
a separate quadratic volume feature. Use the same training data, loss,
and optimizer budget. On the same compression and tension batches,
compare errors in $W$, $P$, and the material tangent with the original model.

#### Encode shape and volume with different growth rates

Keep the same two-hidden-layer MLP and change its input features. First,
take a uniaxial compression $F_e=\operatorname{diag}(s,1,1)$. The first
direction gets shorter as $s\to0^+$ and the other two stay at unit length.
Hence $J=s$ and $I_1=s^2+2$. The original shape feature is

$$z_1=\frac{s^2+2}{s^{2/3}}-3.$$

The two unchanged directions contribute 2 to the numerator while the
denominator tends to zero, so $z_1$ grows without bound. Multiplying
by $J^{2/3}$ cancels that denominator and rescales the subtraction too,

$$J^{2/3}z_1=I_1-3J^{2/3}=s^2+2-3s^{2/3}\longrightarrow2.$$

This rescaled shape feature stays finite in strong compression. For
large tensile stretches on the same path, its leading term is $s^2$.
The code compares the two features as the first stretch approaches zero.

In [ ]:
compression_stretches = np.array([1., .25, .01, 1e-4])
compression_F = torch.from_numpy(uniaxial_path(compression_stretches))
compression_z = invariants(compression_F)
original_shape = compression_z[:, 0]
rescaled_shape = torch.exp((2 / 3) * compression_z[:, 2]) * original_shape

print(f'{"s":>8} {"z1":>12} {"J^(2/3) z1":>14}')
for s, old, scaled in zip(compression_stretches, original_shape, rescaled_shape):
    print(f'{s:>8g} {old:>12.6g} {scaled:>14.6g}')

##### Represent the direction and magnitude of volume change

Set $\ell=\log J$. For $J=0.5$ and $J=2$, $\ell$ has opposite signs
while $\ell^2$ is the same. The signed input lets the network distinguish
compression from expansion. The squared input exposes a quadratic
dependence on the size of the volume change. Neo-Hookean contains
$\tfrac12\lambda(\log J)^2$, so this feature gives the MLP a direct way
to learn such a term from the data.

Replace the second shape input with $\ell^2$ and keep $\ell$ in the
last slot. The new input is $(J^{2/3}z_1,\ell^2,\ell)$.

In [ ]:
def structured_features(Fe):
    z = invariants(Fe)
    log_J = z[..., 2]
    shape = torch.exp((2 / 3) * log_J) * z[..., 0]
    return torch.stack((shape, log_J.square(), log_J), dim=-1)

##### Keep the stress-free reference correction

At $F_e=I$, we have $J=1$, $z_1=0$, $\mathrm dz_1=0$, and $\ell=0$.
By the product rule,

$$\mathrm d(J^{2/3}z_1)=J^{2/3}\,\mathrm dz_1+z_1\,\mathrm d(J^{2/3})=0,$$

and $\mathrm d(\ell^2)=2\ell\,\mathrm d\ell=0$. Only the last input
$\ell$ contributes to the network's first variation. Subtracting
$f_\omega(0)$ sets the rest energy to zero, and subtracting
$\partial_{z_3}f_\omega(0)\,\ell$ cancels that remaining first-order
term. Check the three feature derivatives and then check $W=P=0$
for the complete model at rest.

In [ ]:
structured_feature_gradients = torch.autograd.functional.jacobian(structured_features, rest_Fe)
torch.testing.assert_close(structured_feature_gradients, expected_gradients, rtol=0, atol=1e-12)

torch.manual_seed(42)
structured_model = NeuralEnergy(features=structured_features)
W_rest, P_rest = prediction(structured_model, torch.eye(3, dtype=dtype)[None])
torch.testing.assert_close(W_rest, torch.zeros_like(W_rest), atol=1e-12, rtol=0)
torch.testing.assert_close(P_rest, torch.zeros_like(P_rest), atol=1e-12, rtol=0)

#### Fit with the same training protocol

Start with the same initialization seed and run the same Adam and
L-BFGS stages. Select weights by the in-range validation loss and
freeze them before checking the extrapolation paths. The test paths
enter only after fitting and checkpoint selection.

In [ ]:
structured_checkpoint = {'loss': float('inf'), 'weights': None}
structured_initial = validate(structured_model, structured_checkpoint, 'Initial', 0)
run_adam(structured_model, structured_checkpoint)
run_lbfgs(structured_model, structured_checkpoint)
assert structured_checkpoint['loss'] < structured_initial
structured_model.load_state_dict(structured_checkpoint['weights'])
structured_model.eval().requires_grad_(False)

#### Compare errors on the same three batches

Reuse the exact same in-range, strong-compression, and strong-tension
samples. Each bar is the relative error over the full batch, including
all stress or tangent components. The vertical axis is logarithmic so
the in-range and extrapolation errors fit on the same plot.

In [ ]:
structured_errors = {}
print(f'{"Uniaxial test":<22} {"Model":<12} {"W error":>10} {"P error":>10} {"Tangent":>10}')
for label, stretches in stretch_ranges.items():
    errors = relative_errors(structured_model, uniaxial_path(stretches))
    structured_errors[label] = errors
    for name, (W_error, P_error, H_error) in (
        ('Original', baseline_errors[label]), ('Structured', errors),
    ):
        print(f'{label:<22} {name:<12} {W_error:>10.3%} {P_error:>10.3%} {H_error:>10.3%}')

x = np.arange(len(stretch_ranges))
original_percent = 100 * np.array(list(baseline_errors.values()))
structured_percent = 100 * np.array(list(structured_errors.values()))
fig, axes = plt.subplots(1, 3, figsize=(11, 3.6), layout='constrained')
for i, (ax, title) in enumerate(zip(axes, ('Energy', 'Stress', 'Material tangent'))):
    ax.bar(x - .18, original_percent[:, i], .36, label='Original')
    ax.bar(x + .18, structured_percent[:, i], .36, label='Structured')
    ax.set_xticks(x, ['Compression', 'In range', 'Tension'])
    ax.set(title=title, ylabel='Relative error (%)', yscale='log')
    ax.grid(axis='y', alpha=.25)
axes[0].legend()
plt.show()

In this run, the structured inputs reduce all three errors on both
extrapolation batches. The largest improvement is in compression. The
tangent error falls from about 166% to 6.7%. This comparison covers the
uniaxial path for the Neo-Hookean material used in this lesson.

**Checkpoint** You can train an energy network on local labels, evaluate its
constitutive derivatives, and export its weights for lesson 6.